# Melbourne Liveability Index - Exploratory Data Analysis

This notebook explores the distributions, correlations, and patterns in the liveability metrics.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
sys.path.insert(0, '..')

from ingestion.base import get_db_connection

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

## Load Data

In [ ]:
# Fetch data from database
conn = get_db_connection()
query = """
SELECT
    s.name,
    cs.rate_per_100k,
    ts.stop_count,
    ts.nearest_train_km,
    ss.avg_icsea_score,
    gs.green_pct_of_suburb,
    pp.median_house_price,
    ls.score_total
FROM suburbs s
LEFT JOIN crime_stats cs ON cs.suburb_id = s.id
LEFT JOIN transport_scores ts ON ts.suburb_id = s.id
LEFT JOIN school_scores ss ON ss.suburb_id = s.id
LEFT JOIN greenspace_scores gs ON gs.suburb_id = s.id
LEFT JOIN property_prices pp ON pp.suburb_id = s.id
LEFT JOIN liveability_scores ls ON ls.suburb_id = s.id
"""

df = pd.read_sql_query(query, conn)
conn.close()

print(f"Loaded {len(df)} suburbs")
print(f"\nData shape: {df.shape}")
print(f"\nMissing values:\n{df.isnull().sum()}")

## Data Coverage

In [ ]:
# Calculate coverage percentage
coverage = (df.notna().sum() / len(df) * 100).round(2)

fig, ax = plt.subplots(figsize=(10, 6))
coverage.sort_values().plot(kind='barh', ax=ax, color='steelblue')
ax.set_xlabel('Data Coverage (%)')
ax.set_title('Data Availability by Metric')
ax.axvline(x=90, color='red', linestyle='--', label='90% threshold')
plt.legend()
plt.tight_layout()
plt.show()

print("Data Coverage Summary:")
print(coverage)

## Score Distributions

In [ ]:
# Plot score distribution
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

metrics = ['rate_per_100k', 'stop_count', 'avg_icsea_score', 'green_pct_of_suburb', 'median_house_price']

for idx, metric in enumerate(metrics):
    axes[idx].hist(df[metric].dropna(), bins=30, color='steelblue', edgecolor='black')
    axes[idx].set_title(f'{metric}')
    axes[idx].set_xlabel('Value')
    axes[idx].set_ylabel('Frequency')

# Hide the last subplot
axes[-1].axis('off')

plt.tight_layout()
plt.show()

# Print statistics
print("\nMetric Statistics:")
print(df[metrics].describe().round(2))

## Correlation Analysis

In [ ]:
# Correlation matrix
numeric_cols = df.select_dtypes(include=[np.number]).columns
corr_matrix = df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=ax)
ax.set_title('Metric Correlation Matrix')
plt.tight_layout()
plt.show()

# Find strong correlations (excluding diagonal)
print("\nStrong Correlations (|r| > 0.5):")
for i in range(len(corr_matrix.columns)):
    for j in range(i+1, len(corr_matrix.columns)):
        corr_val = corr_matrix.iloc[i, j]
        if abs(corr_val) > 0.5:
            col1 = corr_matrix.columns[i]
            col2 = corr_matrix.columns[j]
            print(f"{col1} <-> {col2}: {corr_val:.3f}")

## Overall Liveability Distribution

In [ ]:
# Plot score distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(df['score_total'].dropna(), bins=30, color='steelblue', edgecolor='black')
axes[0].set_xlabel('Liveability Score')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of Overall Liveability Scores')
axes[0].axvline(df['score_total'].mean(), color='red', linestyle='--', label=f'Mean: {df["score_total"].mean():.1f}')
axes[0].legend()

# Box plot
axes[1].boxplot(df['score_total'].dropna())
axes[1].set_ylabel('Liveability Score')
axes[1].set_title('Liveability Score Box Plot')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nLiveability Score Statistics:")
print(df['score_total'].describe().round(2))
print(f"\nTop 10 Most Liveable Suburbs:")
print(df.nlargest(10, 'score_total')[['name', 'score_total']].to_string(index=False))
print(f"\nBottom 10 Least Liveable Suburbs:")
print(df.nsmallest(10, 'score_total')[['name', 'score_total']].to_string(index=False))

## Key Insights

In [ ]:
print("\n=== KEY INSIGHTS ===")
print(f"\n1. Data Coverage:")
print(f"   - Best coverage: {coverage.idxmax()} ({coverage.max():.1f}%)")
print(f"   - Worst coverage: {coverage.idxmin()} ({coverage.min():.1f}%)")

print(f"\n2. Liveability Score Range:")
print(f"   - Min: {df['score_total'].min():.1f}")
print(f"   - Max: {df['score_total'].max():.1f}")
print(f"   - Mean: {df['score_total'].mean():.1f}")
print(f"   - Std Dev: {df['score_total'].std():.1f}")

print(f"\n3. Strongest Metric Correlations with Total Score:")
if 'score_total' in corr_matrix.columns:
    score_corr = corr_matrix['score_total'].sort_values(ascending=False)
    for metric, corr_val in score_corr[1:6].items():
        print(f"   - {metric}: {corr_val:.3f}")